In [111]:
import sys

import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pickle

In [112]:
cap = cv.VideoCapture("Videos/carPark.mp4")

In [113]:
try:
    with open("CarParkPos", "rb") as f:
        posList = pickle.load(f)
except:
    sys.exit("You can't continue without loading carPark as posList.")

width, height = 107, 48


In [114]:
def put_text_rect(img, text, pos, scale=3, thickness=3, color_t=(255, 255, 255), color_r=(255, 0, 255),
                  font=cv.FONT_HERSHEY_PLAIN, offset=10, border=None, color_b=(0, 255, 0)):
    ox, oy = pos
    (w, h), _ = cv.getTextSize(text, font, scale, thickness)

    x1, y1, x2, y2 = ox - offset, oy + offset, ox + w + offset, oy - h - offset
    cv.rectangle(img, (x1, y1), (x2, y2), color_r, cv.FILLED)
    if border is not None:
        cv.rectangle(img, (x1, y1), (x2, y2), color_b, border)
    cv.putText(img, text, (ox, oy), font, scale, color_t, thickness)

    return img, [x1, y2, x2, y1]

In [115]:
def car_park_check(img):
    car_park_count = 0
    for pos in posList:
        x, y = pos
        car_park_img = img[y:y + height, x:x + width]
        count_white = cv.countNonZero(car_park_img)

        if count_white < 220:
            car_park_count += 1
            color = (0, 255, 0)
        else:
            color = (0, 0, 255)

        cv.rectangle(frame, pos, (x + width, y + height), color, 4)
        put_text_rect(frame, str(count_white), (x, y + height - 3), scale=1, thickness=2, offset=0, color_r=count_white)
        put_text_rect(frame, f"Free: {car_park_count}/{len(posList)}", (100, 50), scale=3, thickness=5, offset=20,
                      color_r=(0, 200, 0), border=cv.BORDER_ISOLATED)

In [116]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    img_gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    imgBlurred = cv.GaussianBlur(img_gray, (3, 3), 1)
    imgMedian = cv.medianBlur(imgBlurred, 13)
    imgThreshold = cv.adaptiveThreshold(imgMedian, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY_INV, 25, 16)

    car_park_check(imgThreshold)
    cv.imshow("Image", frame)
    if cv.waitKey(16) & 0xFF == ord('q'):
        break

cv.destroyAllWindows()